## 04 - Inference

**Imports and paths**

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path("..").resolve()
with open(PROJECT_ROOT / "config.yml") as f:
    cfg = yaml.safe_load(f)

TARGET = cfg["data"]["target"]
FEATURES = cfg["features"]
VERSION = cfg["experiment"]["version"]
# Best model chosen by 03_training and written to config.yml
BEST_MODEL = cfg["experiment"]["best_model"]["name"]
BEST_MODEL_VERSION = str(cfg["experiment"]["best_model"]["version"])
N_SAMPLES = cfg["inference"]["n_samples"]
RANDOM_STATE = cfg["data"]["random_state"]

RAW_PATH = PROJECT_ROOT / "data" / "Patient_Health_Records_Raw.csv"
CLEAN_PATH = PROJECT_ROOT / cfg["data"]["cleaned_path"]
SYNTHETIC_PATH = PROJECT_ROOT / cfg["inference"]["synthetic_path"]
PREPROCESSORS_PATH = PROJECT_ROOT / cfg["artifacts"]["encoders_dir"] / f"preprocessors_{VERSION}.joblib"
MODELS_PATH = PROJECT_ROOT / cfg["artifacts"]["models_dir"] / f"models_{VERSION}.joblib"

# Placeholder tokens that mean "value not recorded" (same list as 02_preprocessing)
empty = ["", "nan", "none", "null", "n/a", "na", "unknown"]

**4.1 Generate synthetic test data**

In [2]:
# Sample every column independently from the raw values, so the new rows carry the same messy formats
# (words, units, placeholders) the cleaning rules must handle. The target is left out because it is what we predict.
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
rng = np.random.default_rng(RANDOM_STATE)

sample_columns = [c for c in raw.columns if c not in ["Patient_ID", TARGET]]
synthetic = pd.DataFrame({col: rng.choice(raw[col].to_numpy(), size=N_SAMPLES) for col in sample_columns})
# New IDs with an S prefix so synthetic patients can never be confused with real ones
synthetic.insert(0, "Patient_ID", [f"ID-S{i:03d}" for i in range(1, N_SAMPLES + 1)])

SYNTHETIC_PATH.parent.mkdir(parents=True, exist_ok=True)
synthetic.to_csv(SYNTHETIC_PATH, index=False)
print(f"Saved {synthetic.shape} to {SYNTHETIC_PATH.name}")
synthetic.head()

Saved (25, 16) to Synthetic_Test_Data.csv


,Patient_ID,Name,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Last_Visit_Date,Follow_Up,Diagnosis_Code,Notes
0,ID-S001,Jane Doe,-5,Unknown,chicago,22.5,127/80,114,190,N,Former,Lisinopril / Metformin,20250402,N/A,E11.9,
1,ID-S002,John Doe,12,F,Los Angeles,22.5,120 - 80,eighty,250,Yes,n/a,Aspirin; Metformin,24/06/2025,30,,
2,ID-S003,michael brown,29,,New York,22kg/m2,N/A,0,190,no,yes,"aspirin,metformin",May 01 2023,N/A,,Patient feeling fine 🙂
3,ID-S004,"SMITH, ALICE",-5,F,chicago,22kg/m2,115/71,eighty,normal,Unknown,No,,Feb 29 2024,14,I10,
4,ID-S005,"LEE, Sara",-5,FEMALE,New York,N/A,120/80,500,,Unknown,EX-smoker,None,20220130,30,E11.9,Follow up soon


**4.2 Load and validate input**

In [3]:
# Read the file the same way 02_preprocessing reads the raw data, so placeholders are handled identically
df = pd.read_csv(SYNTHETIC_PATH)

# Fail fast if a column the cleaning rules expect is missing
required_columns = [c for c in raw.columns if c != TARGET]
missing_columns = set(required_columns) - set(df.columns)
assert not missing_columns, f"Input is missing columns: {missing_columns}"
print(df.shape)

(25, 16)


**4.3 Fill values from training data**

In [4]:
# Gaps must be filled with the values used in preprocessing, not with statistics of these few new rows.
# Filling with a mode/mean/median leaves that statistic unchanged, so recomputing it on the cleaned dataset
# returns exactly the values 02_preprocessing used.
reference = pd.read_csv(CLEAN_PATH)

FILL_VALUES = {
    "Age": reference["Age"].mode()[0],
    "Gender": reference["Gender"].mode()[0],
    "City": reference["City"].mode()[0],
    "BMI": reference["BMI"].mean(),
    "Blood_Pressure": reference["Blood_Pressure"].mode()[0],
    "Heart_Rate": reference["Heart_Rate"].median(),
    "Cholesterol_Level": reference["Cholesterol_Level"].mode()[0],
    "Diabetic": reference["Diabetic"].mode()[0],
    "Smoker": reference["Smoker"].mode()[0],
    "Medications": "none",
    "Follow_Up": reference["Follow_Up"].mode()[0],
    "Diagnosis_Code": "Unknown",
    "Notes": "No Notes",
}
pd.Series(FILL_VALUES, name="fill_value")

Age                       20.0
Gender                       F
City                  New York
BMI                  23.075909
Blood_Pressure          120/80
Heart_Rate                85.0
Cholesterol_Level        250.0
Diabetic                     N
Smoker                      Ex
Medications               none
Follow_Up                 14.0
Diagnosis_Code         Unknown
Notes                 No Notes
Name: fill_value, dtype: object

**4.4 Cleaning functions**

In [5]:
# Same rules as 02_preprocessing, one function per column

# Ages are recorded as words, x10 entry errors (250) and negative sentinels (-5)
age_words = {'twenty': 20, 'thirty': 30, 'forty': 40, 'fourty': 40, 'fifty': 50}

def clean_age(x):
    x = str(x).strip().lower()
    if x in age_words:
        return age_words[x]
    try:
        val = float(x)
    except ValueError:
        return np.nan
    if val < 0:
        return np.nan
    return val / 10 if val > 120 else val


# Collapse case and abbreviation variants into M/F
gender = {'male': 'M', 'female': 'F', 'm': 'M', 'f': 'F'}

def clean_gender(x):
    x = str(x).lower().strip()
    x = gender.get(x, x)
    return np.nan if x in ['unknown', 'nan', ''] else x


# Merge abbreviations and typos (NY, newyork, Nwe Yrok) into one city name
city = {
    'boston': 'Boston',
    'la': 'Los Angeles',
    'losangeles': 'Los Angeles',
    'newyork': 'New York',
    'nweyrok': 'New York',
    'ny': 'New York',
    'chicago': 'Chicago',
}

def city_standard(x):
    x = str(x).lower().replace(' ', '').strip()
    if x in empty:
        return np.nan
    return city.get(x, x.title())


# Strip the kg/m2 unit so every BMI is a plain number
def BMI_standard(x):
    x = str(x).lower().strip()
    if x in empty:
        return np.nan
    return float(x.replace('kg/m2', ''))


# Unify '120 over 80' and '120 - 80' into the standard 'systolic/diastolic' format
def correct_bp(x):
    x = str(x).lower().strip()
    if x in empty:
        return np.nan
    return x.replace(' over ', '/').replace(' - ', '/')


# Keep numeric rates in a realistic human range; sentinels (0, 500) and words become missing
def clean_heart_rate(x):
    x = str(x).strip().lower()
    if x in empty or not x.replace('.', '').isdigit():
        return np.nan
    val = float(x)
    return val if 40 <= val <= 180 else np.nan


# Convert text levels (high/normal/low) to representative mg/dL values
cholesterol = {'high': '250', 'low': '100', 'normal': '125'}

def correct_cholesterol(x):
    x = str(x).lower().strip()
    if x in empty:
        return np.nan
    return cholesterol.get(x, x)


# Map yes/no variants to Y/N
def clean_diabetic(x):
    x = str(x).strip().lower()
    if x in empty:
        return np.nan
    return {'yes': 'Y', 'y': 'Y', 'no': 'N', 'n': 'N'}.get(x, np.nan)


# Smoking status: Y = current, N = never, Ex = former (EX-smoker/Former)
def clean_smoker(x):
    x = str(x).strip().lower()
    if x in empty:
        return np.nan
    return {'yes': 'Y', 'y': 'Y', 'no': 'N', 'n': 'N', 'ex-smoker': 'Ex', 'former': 'Ex'}.get(x, np.nan)


# Unify list separators (',', ';', '/') so the same drug combination has one spelling
def clean_meds(x):
    x = str(x).lower().strip()
    if x in empty:
        return np.nan
    x = x.replace(';', ',').replace('/', ',')
    return x.replace(' ,', ',').replace(', ', ',')


# Keep follow-up as whole days; text such as 'two weeks' becomes missing
def clean_follow_up(x):
    x = str(x).strip().lower()
    if x in empty:
        return np.nan
    try:
        return int(float(x))
    except ValueError:
        return np.nan


# Placeholder codes are missing
def clean_diagnosis_code(x):
    return np.nan if x in empty else x


# Placeholder notes are missing
def clean_notes(x):
    x = str(x).strip()
    return np.nan if x in empty else x


CLEANERS = {
    "Age": clean_age,
    "Gender": clean_gender,
    "City": city_standard,
    "BMI": BMI_standard,
    "Blood_Pressure": correct_bp,
    "Heart_Rate": clean_heart_rate,
    "Cholesterol_Level": correct_cholesterol,
    "Diabetic": clean_diabetic,
    "Smoker": clean_smoker,
    "Medications": clean_meds,
    "Follow_Up": clean_follow_up,
    "Diagnosis_Code": clean_diagnosis_code,
    "Notes": clean_notes,
}

**4.5 Apply preprocessing transformations**

In [6]:
# Name and Last_Visit_Date are not model features, so only Patient_ID is kept alongside them to label predictions
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    clean = df[["Patient_ID"]].copy()
    for col, cleaner in CLEANERS.items():
        clean[col] = df[col].apply(cleaner).fillna(FILL_VALUES[col])
    # Match the dtypes of the cleaned training data
    clean["Cholesterol_Level"] = clean["Cholesterol_Level"].astype(float)
    clean["Follow_Up"] = clean["Follow_Up"].astype(float)
    return clean


clean = preprocess(df)
assert clean.notna().all().all(), "Missing values remain after cleaning"
clean.head()

,Patient_ID,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Follow_Up,Diagnosis_Code,Notes
0,ID-S001,20.0,F,Chicago,22.500000,127/80,114.0,190.0,N,Ex,"lisinopril,metformin",14.0,E11.9,No Notes
1,ID-S002,12.0,F,Los Angeles,22.500000,120/80,85.0,250.0,Y,Ex,"aspirin,metformin",30.0,Unknown,No Notes
2,ID-S003,29.0,F,New York,22.000000,120/80,85.0,190.0,N,Y,"aspirin,metformin",14.0,Unknown,Patient feeling fine 🙂
3,ID-S004,20.0,F,Chicago,22.000000,115/71,85.0,125.0,N,N,none,14.0,I10,No Notes
4,ID-S005,20.0,F,New York,23.075909,120/80,85.0,250.0,N,Ex,none,30.0,E11.9,Follow up soon


**4.6 Load encoders and model**

In [7]:
# Load the versioned bundles saved by 03_training, so encoders and model always come from the same training run
preprocessors = joblib.load(PREPROCESSORS_PATH)
models = joblib.load(MODELS_PATH)

# Fail fast if config.yml points to a model or version that is not in the saved bundle
assert BEST_MODEL in models, f"{BEST_MODEL} not found in {MODELS_PATH.name}"
assert models[BEST_MODEL]["version"] == BEST_MODEL_VERSION, (
    f"config.yml expects {BEST_MODEL} v{BEST_MODEL_VERSION}, bundle has v{models[BEST_MODEL]['version']}"
)

model = models[BEST_MODEL]["model"]
print(f"Loaded {BEST_MODEL} v{BEST_MODEL_VERSION} from {MODELS_PATH.name}")

Loaded random_forest v1.0.0 from models_v1.joblib


**4.7 Encode features**

In [8]:
# Same steps as transform_features in 03_training, using the encoders fitted on the training split
def transform_features(X: pd.DataFrame) -> pd.DataFrame:
    scaler = preprocessors["standard_scaler"]
    ordinal_encoder = preprocessors["ordinal_encoder"]
    one_hot_encoder = preprocessors["one_hot_encoder"]
    frequency_maps = preprocessors["frequency_maps"]

    scaled = pd.DataFrame(scaler.transform(X[FEATURES["scale"]]), columns=FEATURES["scale"], index=X.index)
    ordinal = pd.DataFrame(ordinal_encoder.transform(X[FEATURES["ordinal"]]), columns=FEATURES["ordinal"], index=X.index)
    # Readings never seen in training get frequency 0
    frequency = pd.DataFrame(
        {f"{col}_freq": X[col].map(frequency_maps[col]).fillna(0.0) for col in FEATURES["frequency"]},
        index=X.index,
    )
    one_hot = pd.DataFrame(
        one_hot_encoder.transform(X[FEATURES["one_hot"]]),
        columns=one_hot_encoder.get_feature_names_out(),
        index=X.index,
    )
    return pd.concat([scaled, ordinal, frequency, one_hot], axis=1)


X_enc = transform_features(clean)

# The model expects exactly the training columns, in the training order
assert list(X_enc.columns) == preprocessors["feature_columns"], "Encoded columns differ from training"
print(X_enc.shape)

(25, 29)


**4.8 Predict**

In [9]:
predictions = pd.DataFrame({
    "Patient_ID": clean["Patient_ID"],
    "predicted_class": model.predict(X_enc),
    # Share of trees voting "disease"; near 0.5 means the model is unsure
    "probability_disease": model.predict_proba(X_enc)[:, 1].round(3),
})
predictions["predicted_label"] = predictions["predicted_class"].map({0: "no disease", 1: "disease"})
predictions

,Patient_ID,predicted_class,probability_disease,predicted_label
0,ID-S001,1,0.550,disease
1,ID-S002,1,0.509,disease
2,ID-S003,1,0.503,disease
3,ID-S004,0,0.475,no disease
4,ID-S005,1,0.507,disease
5,ID-S006,1,0.511,disease
6,ID-S007,1,0.517,disease
7,ID-S008,1,0.504,disease
8,ID-S009,1,0.511,disease
9,ID-S010,0,0.475,no disease
